# 02 — NumPy Core

**What you'll know after this notebook:**
- Create, inspect, and manipulate NumPy arrays
- Index and slice in 1D and 2D
- Reshape, transpose, and stack arrays
- Understand broadcasting — the rule that makes vectorized ML possible
- Do matrix multiplication correctly
- Use reduction operations with the `axis` parameter

**Why this matters:** Every single ML notebook in the exam is built on these operations. If NumPy feels fluent, the algorithms fall into place.

**Time estimate:** 60–90 minutes

In [1]:
import numpy as np

print(np.__version__)

2.5.0


## 1. Creating Arrays

In [2]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print(a)  # [1 2 3 4 5]
print(type(a))  # <class 'numpy.ndarray'>

# 2D array (matrix)
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print(M)

[1 2 3 4 5]
<class 'numpy.ndarray'>
[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [3]:
# Zeros, ones, identity — the building blocks of ML algorithms
print(np.zeros(5))  # [0. 0. 0. 0. 0.]
print(np.zeros((3, 4)))  # 3x4 matrix of zeros
print(np.ones((2, 3)))  # 2x3 matrix of ones
print(np.eye(4))  # 4x4 identity matrix
print(np.full((2, 2), 7))  # 2x2 matrix filled with 7

[0. 0. 0. 0. 0.]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[1. 1. 1.]
 [1. 1. 1.]]
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
[[7 7]
 [7 7]]


In [7]:
# Ranges
print(np.arange(0, 10, 2))  # [0 2 4 6 8] — like Python range()
print(np.linspace(0, 1, 5))  # [0. 0.25 0.5 0.75 1.] — 5 evenly spaced points

# linspace is used for plotting decision boundaries and cost surfaces
x_vals = np.linspace(-5, 5, 100)  # 100 points from -5 to 5

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


In [8]:
# Random arrays — used for weight initialization
np.random.seed(42)  # for reproducibility

print(np.random.rand(3, 2))  # uniform [0, 1)
print(np.random.randn(3, 2))  # standard normal (mean=0, std=1)
print(np.random.randint(0, 10, size=(2, 4)))  # integers in [0, 10)

# Shuffle indices (used for train/test splits)
indices = np.random.permutation(5)
print(indices)

[[0.37454012 0.95071431]
 [0.73199394 0.59865848]
 [0.15601864 0.15599452]]
[[ 1.57921282  0.76743473]
 [-0.46947439  0.54256004]
 [-0.46341769 -0.46572975]]
[[9 5 8 0]
 [9 2 6 3]]
[1 3 4 2 0]


## 2. Array Attributes

Always check shapes before matrix operations — shape mismatch is the #1 source of bugs.

In [9]:
X = np.random.randn(100, 5)  # 100 samples, 5 features

print("shape  :", X.shape)  # (100, 5)
print("ndim   :", X.ndim)  # 2  — number of dimensions
print("size   :", X.size)  # 500 — total elements
print("dtype  :", X.dtype)  # float64

# Access rows and cols separately
m, n = X.shape
print(f"{m} samples, {n} features")

shape  : (100, 5)
ndim   : 2
size   : 500
dtype  : float64
100 samples, 5 features


In [10]:
# CRITICAL: (5,) vs (5,1) vs (1,5) — these are DIFFERENT
a = np.array([1, 2, 3, 4, 5])  # shape (5,) — 1D vector
b = np.array([[1, 2, 3, 4, 5]])  # shape (1,5) — row vector (2D)
c = np.array([[1], [2], [3], [4], [5]])  # shape (5,1) — column vector (2D)

print("a.shape:", a.shape)  # (5,)
print("b.shape:", b.shape)  # (1, 5)
print("c.shape:", c.shape)  # (5, 1)

# (5,) behaves like (5,1) in most ops but can cause bugs
# Use reshape() to be explicit

a.shape: (5,)
b.shape: (1, 5)
c.shape: (5, 1)


## 3. Indexing & Slicing

In [ ]:
# 1D indexing — same as Python lists + negative indexing
v = np.array([10, 20, 30, 40, 50])
print(v[0])  # 10
print(v[-1])  # 50
print(v[1:4])  # [20 30 40]
print(v[::2])  # [10 30 50] — every 2nd element
print(v[::-1])  # [50 40 30 20 10] — reversed

In [11]:
# 2D indexing — [row, col]
M = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print(M[0, 0])  # 1  — top-left
print(M[2, 2])  # 9  — bottom-right
print(M[1, :])  # [4 5 6] — entire row 1
print(M[:, 0])  # [1 4 7] — entire column 0
print(M[0:2, 1:])  # [[2 3], [5 6]] — submatrix

1
9
[4 5 6]
[1 4 7]
[[2 3]
 [5 6]]


In [13]:
# Boolean indexing — selects elements where condition is True
# Used constantly in ML for filtering samples by class label
scores = np.array([45, 82, 91, 34, 76, 88])

mask = scores > 75
print(mask)  # [False True True False True True]
print(scores[mask])  # [82 91 76 88]

# In one line
passing = scores[scores > 75]
print(passing)

[False  True  True False  True  True]
[82 91 76 88]
[82 91 76 88]


In [14]:
# Boolean indexing on 2D — select rows by label
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
y = np.array([0, 1, 0, 1])

X_class0 = X[y == 0]  # rows where label is 0
X_class1 = X[y == 1]  # rows where label is 1
print("Class 0:\n", X_class0)
print("Class 1:\n", X_class1)

Class 0:
 [[1 2]
 [5 6]]
Class 1:
 [[3 4]
 [7 8]]


In [16]:
# Fancy indexing — index with an array of integers
v = np.array([10, 20, 30, 40, 50])
indices = np.array([0, 2, 4])
print(v[indices])  # [10 30 50]

# Used in K-means: get all samples in cluster k
assignments = np.array([0, 1, 0, 2, 1, 0])
data = np.arange(12).reshape(6, 2)
print(data)
cluster0 = data[assignments == 0]
print("Cluster 0:\n", cluster0)

[10 30 50]
[[ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]]
Cluster 0:
 [[ 0  1]
 [ 4  5]
 [10 11]]


## 4. Shape Manipulation

In [12]:
# reshape — change shape without changing data
a = np.arange(12)
print(a)  # [ 0  1  2  3  4  5  6  7  8  9 10 11]
print(a.reshape(3, 4))  # 3x4 matrix
print(a.reshape(4, 3))  # 4x3 matrix
print(a.reshape(2, 2, 3))  # 3D array
print(a.reshape(-1, 3))  # -1 means "figure it out" → 4x3

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
[[[ 0  1  2]
  [ 3  4  5]]

 [[ 6  7  8]
  [ 9 10 11]]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In [17]:
# MNIST use case: 784-pixel image stored as 1D, need to reshape
image_flat = np.random.rand(784)  # 1D: shape (784,)
image_2d = image_flat.reshape(28, 28)  # 2D: shape (28, 28)
print(image_flat.shape, "→", image_2d.shape)

# Batch of 100 images: each is 784 features
images = np.random.rand(100, 784)
images_2d = images.reshape(100, 28, 28)  # (100, 28, 28)
print(images.shape, "→", images_2d.shape)

(784,) → (28, 28)
(100, 784) → (100, 28, 28)


In [18]:
# flatten and ravel — collapse to 1D
M = np.array([[1, 2, 3], [4, 5, 6]])
print(M.flatten())  # [1 2 3 4 5 6] — always returns a copy
print(M.ravel())  # same but may return a view

# Transpose
print(M.shape)  # (2, 3)
print(M.T.shape)  # (3, 2)
print(M.T)

[1 2 3 4 5 6]
[1 2 3 4 5 6]
(2, 3)
(3, 2)
[[1 4]
 [2 5]
 [3 6]]


In [22]:
# np.newaxis — add a dimension (useful for broadcasting)
v = np.array([1, 2, 3])  # shape (3,)

row = v[np.newaxis, :]  # shape (1, 3)
col = v[:, np.newaxis]  # shape (3, 1)

print("v:", v.shape)
print("v:", v.shape)
print("row:", row.shape, row)
print("col:", col.shape, col)

v: (3,)
v: (3,)
row: (1, 3) [[1 2 3]]
col: (3, 1) [[1]
 [2]
 [3]]


## 5. Element-wise Operations & Broadcasting

Broadcasting is *the* rule that makes NumPy ML code concise. Master this once and the rest is easy.

In [ ]:
# Element-wise: same-shape arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print(a + b)  # [5 7 9]
print(a * b)  # [4 10 18]  — element-wise, NOT dot product!
print(a - b)  # [-3 -3 -3]
print(a / b)  # [0.25 0.4 0.5]

In [ ]:
# Broadcasting with a scalar — stretched to match array shape
X = np.array([[1, 2, 3],
              [4, 5, 6]])

print(X * 2)  # every element × 2
print(X + 10)  # every element + 10
print(X ** 2)  # every element squared

In [23]:
# Broadcasting rules:
# Two shapes are compatible if, for each dimension (aligned from the right),
# the sizes are EQUAL or one of them is 1.

#   (3, 4)   +   (4,)   →  (4,) is treated as (1,4) → broadcast to (3,4) ✓
#   (3, 4)   +   (3,1)  →  (3,1) broadcast to (3,4) ✓
#   (3, 4)   +   (3,)   →  ERROR: 3≠4 and neither is 1 ✗

X = np.ones((3, 4))
row_bias = np.array([1, 2, 3, 4])  # shape (4,)
col_bias = np.array([[1], [2], [3]])  # shape (3,1)

print((X + row_bias).shape)  # (3,4)
print((X + col_bias).shape)  # (3,4)

(3, 4)
(3, 4)


In [24]:
# ML application: feature normalization via broadcasting
X = np.array([[1.0, 2000.0, 3.0],
              [2.0, 3000.0, 5.0],
              [3.0, 4000.0, 7.0]])

mu = X.mean(axis=0)  # shape (3,) — mean of each feature column
sigma = X.std(axis=0)  # shape (3,) — std of each feature column

X_norm = (X - mu) / sigma  # broadcast: (3,3) - (3,) / (3,) — each row normalized
print("Normalized X:")
print(X_norm)
print("Means (should be ~0):", X_norm.mean(axis=0))
print("Stds  (should be ~1):", X_norm.std(axis=0))

Normalized X:
[[-1.22474487 -1.22474487 -1.22474487]
 [ 0.          0.          0.        ]
 [ 1.22474487  1.22474487  1.22474487]]
Means (should be ~0): [0. 0. 0.]
Stds  (should be ~1): [1. 1. 1.]


## 6. Matrix Multiplication

This is the core operation of every ML algorithm in the exam.

In [26]:
# @ operator — matrix multiply (Python 3.5+)
# np.dot() is equivalent for 2D arrays

A = np.array([[1, 2], [3, 4], [5, 6]])  # shape (3, 2)
B = np.array([[1, 0, 1], [0, 1, 1]])  # shape (2, 3)

print(A)
print(B)
C = A @ B  # (3,2) @ (2,3) = (3,3)
print("A @ B =")
print(C)
print("shape:", C.shape)

[[1 2]
 [3 4]
 [5 6]]
[[1 0 1]
 [0 1 1]]
A @ B =
[[ 1  2  3]
 [ 3  4  7]
 [ 5  6 11]]
shape: (3, 3)


In [27]:
# @ vs * — CRITICAL DIFFERENCE
a = np.array([[1, 2], [3, 4]])
b = np.array([[2, 0], [1, 3]])

print("a * b (element-wise):")
print(a * b)  # [[2 0], [3 12]]

print("a @ b (matrix multiply):")
print(a @ b)  # [[4 6], [10 12]]

a * b (element-wise):
[[ 2  0]
 [ 3 12]]
a @ b (matrix multiply):
[[ 4  6]
 [10 12]]


In [30]:
# Linear regression prediction: h = X @ W
# X: (m, n+1) — m samples, n features + bias column
# W: (n+1,)   — weights
# h: (m,)     — predictions

m, n = 100, 3
X = np.random.randn(m, n + 1)  # 100 samples, 4 features (incl. bias)
print(X[0:5, :])
X[:, 0] = 1  # bias column = 1
print(X[0:5, :])
W = np.random.randn(n + 1)  # weights
print(W)
h = X @ W  # (100,4) @ (4,) = (100,) — 100 predictions
print("Predictions shape:", h.shape)

[[ 0.98759058 -0.36658333 -0.23350514  1.28571831]
 [ 1.19919948 -0.59887005 -0.39254393 -1.95148512]
 [ 0.55023434  0.57475144 -0.69395177  0.3488509 ]
 [ 0.80801819 -0.97555366 -0.20104745  0.21777843]
 [ 0.51692053  0.58156685 -0.3039287  -0.75854632]]
[[ 1.         -0.36658333 -0.23350514  1.28571831]
 [ 1.         -0.59887005 -0.39254393 -1.95148512]
 [ 1.          0.57475144 -0.69395177  0.3488509 ]
 [ 1.         -0.97555366 -0.20104745  0.21777843]
 [ 1.          0.58156685 -0.3039287  -0.75854632]]
[-0.0965361   0.80649661 -0.02010196 -0.22790521]
Predictions shape: (100,)


In [35]:
# Shape rule: (m, n) @ (n, k) = (m, k)
# Middle dimensions must match!

# Quick check — always do this before multiplying
def can_matmul(A, B):
    return A.shape[-1] == B.shape[-2] if B.ndim >= 2 else A.shape[-1] == B.shape[0]


A = np.ones((5, 3))
B = np.ones((3, 4))
C = np.ones((4, 3))

print(f"{A.shape} @ {B.shape} = {(A @ B).shape}")  # (5,3) @ (3,4) = (5,4)
# print(A @ C)  # ERROR: 3 != 4

(5, 3) @ (3, 4) = (5, 4)


## 7. Reduction Operations with `axis`

In [37]:
# axis=None → reduce entire array to scalar
# axis=0    → reduce across ROWS (collapse rows, keep columns) → result shape is (cols,)
# axis=1    → reduce across COLS (collapse cols, keep rows)    → result shape is (rows,)

X = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]])

print("sum all:", X.sum())  # 45
print("sum axis=0:", X.sum(axis=0))  # [12 15 18] — sum each column
print("sum axis=1:", X.sum(axis=1))  # [ 6 15 24] — sum each row

sum all: 78
sum axis=0: [22 26 30]
sum axis=1: [ 6 15 24 33]


In [38]:
# Same pattern for mean, std, min, max
X = np.random.randn(100, 5)  # 100 samples, 5 features

feature_means = X.mean(axis=0)  # shape (5,) — mean of each feature
sample_norms = X.std(axis=1)  # shape (100,) — std of each sample

print("Feature means shape:", feature_means.shape)
print("Sample norms shape:", sample_norms.shape)

# keepdims=True — keep the reduced axis as size-1 (helps with broadcasting)
col_means = X.mean(axis=0, keepdims=True)  # shape (1, 5)
X_centered = X - col_means  # broadcast: (100,5) - (1,5) = (100,5)

Feature means shape: (5,)
Sample norms shape: (100,)


In [42]:
# argmin / argmax — return INDEX of min/max
costs = np.array([0.9, 0.6, 0.3, 0.7, 0.4])
print("min index:", costs.argmin())  # 2
print("max index:", costs.argmax())  # 0

# In 2D — K-means centroid assignment
# distances[i, k] = distance from sample i to centroid k
distances = np.array([[1.2, 3.5, 0.8],
                      [2.1, 0.3, 1.9],
                      [0.5, 4.2, 2.7],
                      [0.5, 4.2, 2.7]])

assignments = distances.argmin(axis=1)  # closest centroid per sample
print("Cluster assignments:", assignments)  # [2, 1, 0]

min index: 2
max index: 0
Cluster assignments: [2 1 0 0]


## 8. Stacking Arrays

In [44]:
a = np.array([[1, 2], [3, 4]])  # (2, 2)
b = np.array([[5, 6], [7, 8]])  # (2, 2)

# vstack — stack vertically (add rows)
print(np.vstack([a, b]))  # shape (4, 2)

# hstack — stack horizontally (add columns)
print(np.hstack([a, b]))  # shape (2, 4)

# column_stack — stack 1D arrays as columns
x1 = np.array([1, 2, 3])
x2 = np.array([4, 5, 6])
print(np.column_stack([x1, x2]))  # shape (3, 2)

[[1 2]
 [3 4]
 [5 6]
 [7 8]]
[[1 2 5 6]
 [3 4 7 8]]
[[1 4]
 [2 5]
 [3 6]]


In [43]:
# Adding a bias column (column of 1s) — done in EVERY regression notebook
X = np.random.randn(5, 3)  # 5 samples, 3 features

ones = np.ones((5, 1))
X_with_bias = np.hstack([ones, X])  # (5, 4) — bias column first
print(X_with_bias.shape)
print(X_with_bias[:, 0])  # all ones

(5, 4)
[1. 1. 1. 1. 1.]


## 9. Math Functions

In [45]:
# All numpy math functions apply element-wise
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

print(np.exp(x))  # e^x
print(np.log(np.exp(x)))  # natural log — inverse of exp
print(np.sqrt(np.abs(x)))  # sqrt(|x|)
print(np.abs(x))  # absolute value
print(np.clip(x, -1, 1))  # clamp to [-1, 1]

[0.13533528 0.36787944 1.         2.71828183 7.3890561 ]
[-2. -1.  0.  1.  2.]
[1.41421356 1.         0.         1.         1.41421356]
[2. 1. 0. 1. 2.]
[-1. -1.  0.  1.  1.]


In [46]:
# Sigmoid function — the activation used in logistic regression + neural networks
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


z = np.array([-10, -2, 0, 2, 10])
print(sigmoid(z))  # [~0, 0.12, 0.5, 0.88, ~1]

# Works on scalars, 1D and 2D arrays — thanks to NumPy broadcasting
print(sigmoid(0.0))  # scalar
print(sigmoid(np.ones((3, 2))))  # 2D array

[4.53978687e-05 1.19202922e-01 5.00000000e-01 8.80797078e-01
 9.99954602e-01]
0.5
[[0.73105858 0.73105858]
 [0.73105858 0.73105858]
 [0.73105858 0.73105858]]


In [ ]:
# np.log is natural log — used in cross-entropy loss
# Be careful with log(0) = -inf — always clip predictions
y_hat = np.array([0.9, 0.1, 0.8, 0.2])
y = np.array([1, 0, 1, 0])

# Binary cross-entropy
epsilon = 1e-8  # prevent log(0)
loss = -np.mean(y * np.log(y_hat + epsilon) + (1 - y) * np.log(1 - y_hat + epsilon))
print(f"Cross-entropy loss: {loss:.4f}")

## 10. Linear Algebra

In [47]:
# np.linalg.norm — Euclidean distance (used in K-means)
a = np.array([3.0, 4.0])
print(np.linalg.norm(a))  # 5.0 — ||a||₂

# Distance between two points
p1 = np.array([1.0, 2.0])
p2 = np.array([4.0, 6.0])
print(np.linalg.norm(p1 - p2))  # 5.0

# Norm over rows or columns
X = np.array([[3, 4], [1, 0], [0, 2]])
print(np.linalg.norm(X, axis=1))  # norm of each row: [5, 1, 2]

5.0
5.0
[5. 1. 2.]


In [49]:
# np.linalg.eig — eigenvalue decomposition (used in PCA)
Sigma = np.array([[4, 2],
                  [2, 3]])  # covariance matrix (must be symmetric)

eigenvalues, eigenvectors = np.linalg.eig(Sigma)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

# Sort by eigenvalue descending (largest variance first)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]  # sort COLUMNS
print("\nSorted eigenvalues:", eigenvalues)

Eigenvalues: [5.56155281+0.j 1.43844719+0.j]
Eigenvectors (columns):
 [[ 0.78820544+0.j -0.61541221+0.j]
 [ 0.61541221+0.j  0.78820544+0.j]]

Sorted eigenvalues: [5.56155281+0.j 1.43844719+0.j]
Eigenvectors (columns):
 [[ 0.78820544+0.j -0.61541221+0.j]
 [ 0.61541221+0.j  0.78820544+0.j]]


In [53]:
# np.linalg.inv — matrix inverse (used in Normal Equation of linear regression)
A = np.array([[2.0, 1.0], [1.0, 3.0]])
A_inv = np.linalg.inv(A)
print("A @ A_inv =")
print(np.round(A @ A_inv, 10))  # should be identity matrix

# Normal equation: W = (X^T X)^(-1) X^T y
# In practice use gradient descent instead (more scalable)

A @ A_inv =
[[ 1.  0.]
 [-0.  1.]]


---
## Exercises

These exercises replicate the exact NumPy patterns used in the exam notebooks.

### Exercise 1: Shape Detective
Without running the code, write down the resulting shape of each operation.

In [55]:
A = np.ones((4, 3))
B = np.ones((3, 5))
v = np.ones(3)

# Predict shapes:
print((A @ B).shape)  # (4, 5)
print((A + v).shape)  # ?  (broadcast)
print((A.T).shape)  # ?
print(A.reshape(2, -1).shape)  # ?
print(np.hstack([A, A]).shape)  # ?

(4, 5)
(4, 3)
(3, 4)
(2, 6)
(4, 6)


### Exercise 2: Feature Normalization
Implement `feature_normalize(X)` that returns X with each feature (column) standardized to zero mean and unit variance. Must work for any shape `(m, n)` using broadcasting — no loops.

In [56]:
def feature_normalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm


X = np.array([[1.0, 200.0],
              [2.0, 400.0],
              [3.0, 600.0],
              [4.0, 800.0]])

X_norm = feature_normalize(X)
print(X_norm)
print("Means (should be ~0):", X_norm.mean(axis=0))
print("Stds  (should be ~1):", X_norm.std(axis=0))

[[-1.34164079 -1.34164079]
 [-0.4472136  -0.4472136 ]
 [ 0.4472136   0.4472136 ]
 [ 1.34164079  1.34164079]]
Means (should be ~0): [0. 0.]
Stds  (should be ~1): [1. 1.]


In [57]:
# SOLUTION
def feature_normalize(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    return (X - mu) / sigma


X = np.array([[1.0, 200.0], [2.0, 400.0], [3.0, 600.0], [4.0, 800.0]])
X_norm = feature_normalize(X)
print(X_norm)
print("Means:", X_norm.mean(axis=0))
print("Stds: ", X_norm.std(axis=0))

[[-1.34164079 -1.34164079]
 [-0.4472136  -0.4472136 ]
 [ 0.4472136   0.4472136 ]
 [ 1.34164079  1.34164079]]
Means: [0. 0.]
Stds:  [1. 1.]


### Exercise 3: Vectorized MSE
Implement `compute_cost(X, y, W)` using vectorized NumPy operations.
MSE = `(1/(2m)) * sum((X @ W - y)^2)`

In [76]:
def compute_cost(X, y, W):
    m = len(y)
    MSE = (1 / (2 * m)) * np.sum((X @ W - y) ** 2)
    return MSE


np.random.seed(42)
m, n = 50, 3
X = np.hstack([np.ones((m, 1)), np.random.randn(m, n)])
print(X.shape)
y = np.random.randn(m)
print(y.shape)
W = np.zeros(n + 1)
print(W.shape)
print(compute_cost(X, y, W))  # should be a scalar

(50, 4)
(50,)
(4,)
0.3948936622258854


In [74]:
# SOLUTION
def compute_cost(X, y, W):
    m = len(y)
    error = X @ W - y
    return (1 / (2 * m)) * (error @ error)


print(compute_cost(X, y, W))

0.3948936622258854


### Exercise 4: Boolean Masking
Given `X` (samples) and `y` (binary labels 0 or 1), return a dict with keys `0` and `1`, each containing the subset of `X` for that class.

In [82]:
def split_by_class(X, y):
    zero = X[y == 0]
    one = X[y == 1]
    print(zero)
    print(one)
    return {0: zero, 1: one}


np.random.seed(0)
X = np.random.randn(10, 2)
y = np.array([0, 1, 0, 1, 1, 0, 0, 1, 1, 0])

classes = split_by_class(X, y)
print("Class 0 shape:", classes[0].shape)  # (5, 2)
print("Class 1 shape:", classes[1].shape)  # (5, 2)

[[ 1.76405235  0.40015721]
 [ 1.86755799 -0.97727788]
 [ 0.14404357  1.45427351]
 [ 0.76103773  0.12167502]
 [ 0.3130677  -0.85409574]]
[[ 0.97873798  2.2408932 ]
 [ 0.95008842 -0.15135721]
 [-0.10321885  0.4105985 ]
 [ 0.44386323  0.33367433]
 [ 1.49407907 -0.20515826]]
Class 0 shape: (5, 2)
Class 1 shape: (5, 2)


In [83]:
# SOLUTION
def split_by_class(X, y):
    return {label: X[y == label] for label in np.unique(y)}


classes = split_by_class(X, y)
print("Class 0 shape:", classes[0].shape)
print("Class 1 shape:", classes[1].shape)

Class 0 shape: (5, 2)
Class 1 shape: (5, 2)


### Exercise 5: One-Hot Encoding (NumPy style)
Implement `to_one_hot(y, num_classes)` that converts a 1D array of integer labels into a 2D one-hot matrix. Shape: `(m, num_classes)`. Use `np.zeros` and integer indexing — no loops.

In [ ]:
def to_one_hot(y, num_classes):
    # YOUR CODE HERE
    pass


y = np.array([0, 2, 1, 3, 1])
print(to_one_hot(y, 4))
# Expected:
# [[1 0 0 0]
#  [0 0 1 0]
#  [0 1 0 0]
#  [0 0 0 1]
#  [0 1 0 0]]

In [84]:
# SOLUTION
def to_one_hot(y, num_classes):
    m = len(y)
    result = np.zeros((m, num_classes))
    result[np.arange(m), y] = 1  # fancy indexing: row i, col y[i]
    return result


y = np.array([0, 2, 1, 3, 1])
print(to_one_hot(y, 4))

[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]


### Exercise 6: Gradient Descent Step
Implement one step of gradient descent for linear regression:
- gradient = `(1/m) * X.T @ (X @ W - y)`
- `W_new = W - lr * gradient`

Return `W_new`.

In [87]:
def gradient_step(X, y, W, lr=0.01):
    m = len(y)
    error = X@W - y
    grad = (1/m) * (np.transpose(X) @ error)
    W -= lr * grad
    return W


np.random.seed(0)
m, n = 20, 2
X = np.hstack([np.ones((m, 1)), np.random.randn(m, n)])
print(X.shape)
W_true = np.array([1.0, 2.0, -1.0])
print(W_true.shape)
y = X @ W_true + 0.1 * np.random.randn(m)
print(y.shape)
W = np.zeros(3)
for _ in range(1000):
    W = gradient_step(X, y, W, lr=0.1)

print("Estimated W:", W)
print("True W     :", W_true)

(20, 3)
(3,)
(20,)
Estimated W: [ 0.96814117  1.98756177 -1.01116207]
True W     : [ 1.  2. -1.]


In [88]:
# SOLUTION
def gradient_step(X, y, W, lr=0.01):
    m = len(y)
    error = X @ W - y
    gradient = (1 / m) * (X.T @ error)
    return W - lr * gradient


W = np.zeros(3)
for _ in range(1000):
    W = gradient_step(X, y, W, lr=0.1)

print("Estimated W:", W)
print("True W     :", W_true)

Estimated W: [ 0.96814117  1.98756177 -1.01116207]
True W     : [ 1.  2. -1.]


### Exercises 7–15: Spot the Shape Bug
Each cell has a shape error. Find and fix it.

In [92]:
# Ex 7: Fix the matrix multiply
X = np.ones((100, 4))
W = np.ones((4,))  # BUG: should work for X @ W
result = X @ W
print(result)

[4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.
 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.
 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.
 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.
 4. 4. 4. 4.]


In [94]:
# Ex 8: Fix the normalization
X = np.random.randn(50, 3)
mu = X.mean(axis=0)  # BUG: wrong axis
X_norm = X - mu  # shape mismatch
print(X_norm)
# Fix: change axis

[[ 1.16101786  0.38097592  0.87745635]
 [-0.21558014 -0.88068855 -0.31892131]
 [ 0.23381854  0.44290944  2.35043734]
 [ 0.2085818  -0.8921873  -0.25485339]
 [-0.21275702  0.54523917 -1.44966863]
 [ 0.31410095  0.22026424  0.32330942]
 [-0.34647711 -0.17416403 -1.33293252]
 [-0.24248093 -0.47910377  0.50717843]
 [-0.90534348  0.8449558   1.58561293]
 [-1.81914607  0.49001643  0.76803642]
 [-0.38659807 -0.33351411 -0.04175219]
 [-0.04695192 -0.24525527 -1.58487542]
 [ 1.40317052  1.14337629 -0.72223587]
 [-1.21558537  0.58482258 -0.48465958]
 [ 0.39279212 -0.25557072  0.78266714]
 [ 0.9455881  -0.66183968 -1.29223557]
 [-1.33209944  0.67413708 -1.09773087]
 [-0.2559774  -0.53255634  0.03856109]
 [-1.68544085  0.2525363   0.61501941]
 [ 0.33926104 -0.24712847  0.18852855]
 [ 0.6498853  -2.70883506  2.0470407 ]
 [ 0.64093228 -0.58865088 -0.29982499]
 [ 0.74458073 -0.05234624 -1.93955608]
 [ 2.31533182 -0.04678296  1.1113011 ]
 [-0.44121089  1.60013476  0.37747208]
 [ 0.85968279 -0.98149567

In [95]:
# Ex 9: Fix the one-hot assignment
m = 5
y = np.array([0, 2, 1, 3, 1])
result = np.zeros((m, 4))
result[np.arange(m), y] = 1  # BUG: wrong indexing
print(result)
# Fix: use np.arange(m) as row index

[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]


In [ ]:
# SOLUTIONS for 7-9
# Ex 7
W = np.ones((4,))  # fix: W must have 4 elements to match X's 4 columns
result = X @ W
print("Ex7 result shape:", result.shape)  # (100,)

# Ex 8
X = np.random.randn(50, 3)
mu = X.mean(axis=0)  # fix: axis=0 gives shape (3,) — one mean per feature
X_norm = X - mu
print("Ex8 norm shape:", X_norm.shape)  # (50, 3)

# Ex 9
result = np.zeros((5, 4))
result[np.arange(5), y] = 1  # fix: index row AND column
print("Ex9 one-hot:\n", result)

---
## Summary

| Operation | Syntax | Notes |
|-----------|--------|-------|
| Create | `np.zeros((m,n))`, `np.ones`, `np.eye`, `np.random.randn` | Always specify shape as tuple |
| Inspect | `.shape`, `.dtype`, `.ndim` | Check before every matmul |
| Index 2D | `A[row, col]`, `A[:, 0]`, `A[y==k]` | Comma not bracket |
| Reshape | `.reshape(m, n)`, `-1` for auto | `.T` for transpose |
| Matmul | `A @ B` | Inner dims must match |
| Element-wise | `A * B`, `A + B` | NOT matmul |
| Reduce | `.sum(axis=0)`, `.mean(axis=1)` | 0=collapse rows, 1=collapse cols |
| argmin | `.argmin(axis=1)` | Returns index, not value |
| Stack | `np.hstack`, `np.vstack`, `np.column_stack` | |
| Math | `np.exp`, `np.log`, `np.sqrt`, `np.abs` | All element-wise |
| Linalg | `np.linalg.eig`, `np.linalg.norm`, `np.linalg.inv` | |

**Next:** [03_numpy_advanced_ml.ipynb](03_numpy_advanced_ml.ipynb) — implement the actual ML building blocks.